# Fine-tune Ministral-8B on dbt DAG generation

**Goal**: teach Ministral-8B-Instruct to take a business question + SQL schemas → produce a  
production-ready multi-file dbt DAG (staging + optional intermediate + marts).

**Technique**: QLoRA (4-bit quantisation + LoRA adapters) with `unsloth` for fast training.

**Runtime**: GPU — use **T4 (free)** or **A100 (Colab Pro)** for faster runs.

---
### Steps
1. Install dependencies
2. Upload dataset files
3. Load model in 4-bit
4. Attach LoRA adapters
5. Train with SFTTrainer
6. Evaluate on held-out set
7. Push merged model to HuggingFace Hub
8. Export GGUF for LM Studio

## 0 — Runtime check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — change Runtime → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 — Install dependencies

In [ ]:
# unsloth: optimised QLoRA trainer for Mistral/Llama families
# Install the correct wheel for the current CUDA version
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]",
    "trl>=0.9",
    "transformers>=4.40",
    "datasets",
    "bitsandbytes",
    "peft",
    "accelerate",
    "-q",
], check=True)
print("Dependencies installed.")

## 2 — Upload dataset & configuration

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)

print("Upload train.jsonl and eval.jsonl from 02_finetuning/data/")
uploaded = files.upload()
for fname, data in uploaded.items():
    path = f'data/{fname}' if not fname.startswith('data/') else fname
    with open(path, 'wb') as f:
        f.write(data)
    print(f"  Saved → {path}")

# Verify
import json
train_rows = [json.loads(l) for l in open('data/train.jsonl')]
eval_rows  = [json.loads(l) for l in open('data/eval.jsonl')]
print(f"\nTrain: {len(train_rows)} rows  |  Eval: {len(eval_rows)} rows")
print("Sample keys:", list(train_rows[0].keys()))
print("Sample roles:", [m['role'] for m in train_rows[0]['messages']])

## 3 — Load Ministral-8B in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME    = "mistralai/Ministral-8B-Instruct-2410"
MAX_SEQ_LEN   = 2048   # all rows fit under ~950 tokens; headroom for prompt
LOAD_IN_4BIT  = True
DTYPE         = None   # auto-detect (bfloat16 on Ampere+, float16 on T4)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 4 — Attach LoRA adapters

In [ ]:
# ── LoRA config ─────────────────────────────────────────────────────────────
# r=16 is a good default: more expressive than r=8, less VRAM than r=64
# Target all attention + feed-forward projection layers
LORA_R         = 16
LORA_ALPHA     = 32    # scaling = alpha / r = 2; higher = larger effective LR
LORA_DROPOUT   = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention
    "gate_proj", "up_proj", "down_proj",        # MLP
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # significantly reduces VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

## 5 — Prepare datasets

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments
import json

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

train_data = Dataset.from_list(load_jsonl('data/train.jsonl'))
eval_data  = Dataset.from_list(load_jsonl('data/eval.jsonl'))

# Format messages into the Mistral instruct template
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_data = train_data.map(apply_chat_template)
eval_data  = eval_data.map(apply_chat_template)

print("Sample formatted text (first 600 chars):")
print(train_data[0]['text'][:600])

## 6 — Train

In [ ]:
from unsloth import is_bfloat16_supported

# ── Training hyperparameters ─────────────────────────────────────────────────
# With 900 rows and batch_size=4, grad_accum=4 → effective batch = 16
# 3 epochs * 900 / 16 ≈ 168 steps  → fast even on T4

training_args = TrainingArguments(
    output_dir                  = "./checkpoints",
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.05,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    optim                       = "adamw_8bit",  # 8-bit Adam saves VRAM
    seed                        = 42,
    report_to                   = "none",
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_data,
    eval_dataset    = eval_data,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ_LEN,
    args            = training_args,
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nDone. Loss: {trainer_stats.training_loss:.4f}")

## 7 — Quick quality check (inference)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # 2x faster inference

SYSTEM_PROMPT = trainer.train_dataset[0]['text'].split('[INST]')[0]  # reuse from data

test_question = "Show the total revenue per product category."
test_context  = "CREATE TABLE products (product_id INT, category VARCHAR, price DECIMAL); CREATE TABLE orders (order_id INT, product_id INT, quantity INT)"

messages = [
    {"role": "system",  "content": "You are a Staff Analytics Engineer expert in dbt."},
    {"role": "user",    "content": f"Business question: {test_question}\n\nSQL schemas:\n{test_context}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=1024,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(generated)

## 8 — Save LoRA adapters (lightweight checkpoint)

In [ ]:
# Save LoRA weights only (~60 MB for r=16) — fast, good for resuming
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
print("LoRA adapters saved to ./lora_adapters")

# Optional: download the adapters to your machine for later use or HF upload
# from google.colab import files
# import shutil
# shutil.make_archive('lora_adapters', 'zip', 'lora_adapters')
# files.download('lora_adapters.zip')

## 9 — Export GGUF for LM Studio

This merges the LoRA weights into the base model and quantises to GGUF Q4_K_M —  
the format LM Studio loads directly.

In [ ]:
# Merge LoRA into base model and save as GGUF Q4_K_M
# Q4_K_M = 4-bit quantisation, good quality/size balance (~4.5 GB for 8B model)
model.save_pretrained_gguf(
    "ministral-dbt-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF file saved to ./ministral-dbt-gguf/")

import os
for f in os.listdir('ministral-dbt-gguf'):
    size_mb = os.path.getsize(f'ministral-dbt-gguf/{f}') / 1e6
    print(f"  {f}  ({size_mb:.0f} MB)")

In [ ]:
# Download the .gguf file — then load it in LM Studio
from google.colab import files
import glob

gguf_files = glob.glob('ministral-dbt-gguf/*.gguf')
print(f"Found {len(gguf_files)} GGUF file(s)")
for f in gguf_files:
    print(f"Downloading {f} ...")
    files.download(f)

## 10 — (Optional) Push to HuggingFace Hub

Saves the LoRA adapters to your HuggingFace account so you can reuse them later  
without re-training (e.g. try different base models, share with team).

In [ ]:
# Set your HuggingFace token (get it at hf.co/settings/tokens)
HF_TOKEN   = "hf_..."  # replace with your token
HF_REPO_ID = "your-username/ministral-8b-dbt-instruct"  # replace with your repo

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")